In [14]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import numpy as np


In [15]:
df=pd.read_csv('svr.csv')
df.head()

,Applicant_ID,Age,Annual_Income,Credit_Score,Employment_Years,Debt_to_Income,Existing_Loans,Monthly_Expenses,Property_Value,Dependents,Education,Employment_Type,Property_Type,Loan_Term,Loan_Amount
0,L001,24,38000,642,2,0.42,1,22000,1800000,0,Graduate,Salaried,Rented,5 Years,120000
1,L002,31,72000,735,7,0.28,0,30000,4200000,2,Postgraduate,Salaried,Owned,10 Years,350000
2,L003,45,56000,615,12,0.51,3,38000,2800000,3,Graduate,Self_Employed,Owned,7 Years,280000
3,L004,28,48000,690,4,0.35,1,26000,2200000,1,Graduate,Salaried,Rented,5 Years,190000
4,L005,52,95000,778,20,0.22,0,42000,7500000,2,Postgraduate,Business,Owned,15 Years,650000


In [16]:
df = df.drop("Applicant_ID", axis=1)

In [17]:
X = df.drop("Loan_Amount", axis=1)

y = df["Loan_Amount"]

In [18]:
numeric_features = [
    "Age",
    "Annual_Income",
    "Credit_Score",
    "Employment_Years",
    "Debt_to_Income",
    "Existing_Loans",
    "Monthly_Expenses",
    "Property_Value",
    "Dependents"
]

categorical_features = [
    "Education",
    "Employment_Type",
    "Property_Type",
    "Loan_Term"
]


In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [20]:
svr = SVR(
    kernel="rbf",
    C=10000,
    gamma="scale",
    epsilon=0.1
)


In [21]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("svr", svr)
])


In [22]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [23]:

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['Age','Annual_Income','Credit_Score',...,'Employment_Type', 'Property_Type','Loan_Term']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This s

In [25]:
y_pred = model.predict(X_test)
y_pred

array([240578.51399158, 263434.7044589 , 241611.66576598, 243755.90338695,
       280517.86636392, 239192.97983495, 234998.46092836, 307942.19626782,
       371409.68637902, 266638.91635788, 400998.44484334, 242694.56442859])

In [26]:

print("Actual:")
print(y_test.values)

print("\nPredicted:")
print(y_pred)

print("\nR2 Score:", r2_score(y_test, y_pred))

print(
    "MAE:",
    mean_absolute_error(y_test, y_pred)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_test, y_pred))
)

Actual:
[120000 240000  75000  65000 260000  70000 115000 290000 420000 240000
 750000 160000]

Predicted:
[240578.51399158 263434.7044589  241611.66576598 243755.90338695
 280517.86636392 239192.97983495 234998.46092836 307942.19626782
 371409.68637902 266638.91635788 400998.44484334 242694.56442859]

R2 Score: 0.4013928019434271
MAE: 110329.80338021215
RMSE: 144418.7807507623
